In [1]:
import pandas as pd

df = pd.read_csv('data/action-class-occurrences.csv')

In [4]:
df = pd.read_csv('data/nonviolent-action-classes.csv', sep=";")

In [6]:
import cv2
import mediapipe as mp

VIDEO_PATH = "data/violent/cam1/1.mp4"
cap = cv2.VideoCapture(VIDEO_PATH)

mp_face = mp.solutions.face_detection.FaceDetection(
    model_selection=0, min_detection_confidence=0.6
)

paused = False

while cap.isOpened():
    if not paused:
        ret, frame = cap.read()
        if not ret:  # video ended → restart
            cap.release()
            cap = cv2.VideoCapture(VIDEO_PATH)
            continue

        # Run detection only when advancing frames
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = mp_face.process(rgb)

        if result.detections:
            for det in result.detections:
                bbox = det.location_data.relative_bounding_box
                h, w, _ = frame.shape
                x1, y1 = int(bbox.xmin * w), int(bbox.ymin * h)
                x2, y2 = x1 + int(bbox.width * w), y1 + int(bbox.height * h)
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

    # Always show the last frame (even when paused)
    if 'frame' in locals():
        cv2.imshow("Video - Space: Pause/Resume, F/B: Seek, Q: Quit", frame)

    # Handle keys
    key = cv2.waitKey(30) & 0xFF
    if key == ord("q"):  # quit
        break
    elif key == ord("f"):  # forward 30 frames
        pos = int(cap.get(cv2.CAP_PROP_POS_FRAMES))
        cap.set(cv2.CAP_PROP_POS_FRAMES, pos + 30)
    elif key == ord("b"):  # backward 30 frames
        pos = inqt(cap.get(cv2.CAP_PROP_POS_FRAMES))
        cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, pos - 30))
    elif key == ord(" "):  # spacebar toggles pause
        paused = not paused
        print("Paused" if paused else "Resumed")

cap.release()
cv2.destroyAllWindows()


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1757430963.106932   11794 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in "/home/idio/anaconda3/lib/python3.12/site-packages/cv2/qt/plugins"
QObject::moveToThread: Current thread (0x3d1e03f0) is not the object's thread (0x3d702120).
Cannot move to target thread (0x3d1e03f0)

QObject::moveToThread: Current thread (0x3d1e03f0) is not the object's thread (0x3d702120).
Cannot move to target thread (0x3d1e03f0)

QObject::moveToThread: Current thread (0x3d1e03f0) is not the object's thread (0x3d702120).
Cannot move to target thread (0x3d1e03f0)

QObject::moveToThread: Current thread (0x3d1e03f0) is not the object's thread (0x3d702120).
Cannot move to target thread (0x3d1e03f0)

QObject::moveToThread: Current thread (0x3d1e03f0) is not the object's thread (0x3d702120

In [8]:
import cv2
import mediapipe as mp
import math
import numpy as np

# Initialize MediaPipe Pose
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

# --- Helper function to calculate joint angles ---
def calculate_angle(a, b, c):
    """
    Calculates the angle between three points a, b, c.
    Point 'b' is the vertex (e.g., the elbow).
    a, b, c are lists of [x, y] coordinates.
    """
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)

    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians * 180.0 / np.pi)

    if angle > 180.0:
        angle = 360 - angle

    return angle

pose = mp_pose.Pose(
    static_image_mode=False,        # For video
    model_complexity=2,             # 0=Light, 1=Medium, 2=Heavy (most accurate)
    smooth_landmarks=True,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7)

VIDEO_PATH = "data/violent/cam1/1.mp4"
cap = cv2.VideoCapture(VIDEO_PATH)

paused = False

while cap.isOpened():
    if not paused:
        ret, frame = cap.read()
        if not ret:
            cap.release()
            cap = cv2.VideoCapture(VIDEO_PATH)
            continue

        # Convert the BGR image to RGB and process it with MediaPipe Pose
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(rgb_frame)

        # Draw the pose annotation on the frame.
        if results.pose_landmarks:
            # This draws the full connected skeleton
            mp_drawing.draw_landmarks(
                frame,
                results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS,
                landmark_drawing_spec=mp_drawing_styles.get_default_pose_landmarks_style())

            # --- Optional: Add additional calculations here ---
            # Example: Calculate the angle of the left elbow for analysis
            landmarks = results.pose_landmarks.landmark
            # Get keypoints for shoulder, elbow, wrist
            shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
            elbow = [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x, landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y]
            wrist = [landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x, landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y]

            # Calculate angle function (implementation below)
            angle = calculate_angle(shoulder, elbow, wrist)
            # Display the angle on the frame near the elbow
            h, w, _ = frame.shape
            elbow_px = (int(elbow[0] * w), int(elbow[1] * h))
            cv2.putText(frame, f"{angle:.1f}°", elbow_px, cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

    # Display the frame
    cv2.imshow("Violence Detection - Pose Analysis | Space: Pause/Resume, Q: Quit", frame)

    # Handle keys
    key = cv2.waitKey(30) & 0xFF
    if key == ord("q"):
        break
    elif key == ord(" "):
        paused = not paused
        print("Paused" if paused else "Resumed")

cap.release()
cv2.destroyAllWindows()
pose.close() # Release the Pose model


W0000 00:00:1757431681.649637   15844 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1757431681.754263   15846 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
QObject::moveToThread: Current thread (0x3d1e03f0) is not the object's thread (0x3d702120).
Cannot move to target thread (0x3d1e03f0)

QObject::moveToThread: Current thread (0x3d1e03f0) is not the object's thread (0x3d702120).
Cannot move to target thread (0x3d1e03f0)

QObject::moveToThread: Current thread (0x3d1e03f0) is not the object's thread (0x3d702120).
Cannot move to target thread (0x3d1e03f0)

QObject::moveToThread: Current thread (0x3d1e03f0) is not the object's thread (0x3d702120).
Cannot move to target thread (0x3d1e03f0)

QObject::moveToThread: Current thread (0x3d1e03f0) is not the object's thread (0x3d702120).
Cann